# Branch Predictor Analysis — tinymlp & matmul

Parses gem5 `stats.txt` and `config.ini` for 4 predictors × 2 workloads.  
Generates comparison tables, IPC / misprediction bar charts, LABP loop predictor activity, and storage-efficiency analysis.

**Predictors:** BiModeBP · GshareBP · LAP (LABP) · TAGE  
**Workloads:** tinymlp (data-dependent sparsity branches) · matmul (clean counted loops)

In [1]:
import re
from pathlib import Path
import configparser

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', '{:.4f}'.format)

In [ ]:
PROJECT_ROOT = Path('..').resolve()
RESULTS_DIR  = PROJECT_ROOT / 'results'

WORKLOADS  = ['tinymlp', 'matmul']
PREDICTORS = ['BiModeBP', 'GshareBP', 'LAP', 'TAGE']

PRED_COLORS = {
    'BiModeBP': '#4c72b0',
    'GshareBP': '#dd8452',
    'LAP':      '#55a868',
    'TAGE':     '#c44e52',
}

print('Project root:', PROJECT_ROOT)
print('Results dir: ', RESULTS_DIR)
print('Workloads:   ', WORKLOADS)
print('Predictors:  ', PREDICTORS)

In [ ]:
# ---------- stats.txt parser ----------
_STAT_RE = re.compile(r'^([A-Za-z0-9_.:]+)\s+([-+0-9.eE]+)\s+#')

def load_stats(path):
    out = {}
    with open(path) as fh:
        for line in fh:
            m = _STAT_RE.match(line.strip())
            if m:
                try:
                    out[m.group(1)] = float(m.group(2))
                except ValueError:
                    pass
    return out


# ---------- config.ini helpers ----------
def load_cfg(path):
    cfg = configparser.ConfigParser(interpolation=None)
    cfg.read(path)
    return cfg

def _ci(cfg, sec, key, default=0):
    try:    return int(cfg[sec][key])
    except: return default  # noqa: E722

def _cs(cfg, sec, key, default=None):
    try:    return cfg[sec][key]
    except: return default  # noqa: E722

def _cil(cfg, sec, key, default=None):
    try:
        raw = cfg[sec][key].strip()
        return [int(x) for x in raw.split()] if raw else default
    except:
        return default  # noqa: E722


# ---------- branch counter stat name variants ----------
_LOOKUP_KEYS = [
    'system.cpu.branchPred.condPredicted',
    'system.cpu.branchPred.condLookups',
    'system.cpu.branchPred.lookups',
]
_MISPRED_KEYS = [
    'system.cpu.branchPred.condIncorrect',
    'system.cpu.branchPred.incorrect',
    'system.cpu.branchPred.condMispredicted',
]

def pick_stat(stats, keys):
    for k in keys:
        if k in stats:
            return stats[k]
    return None


# ---------- loop predictor stat prefix ----------
_LP_PREFIX = 'system.cpu.branchPred.conditionalBranchPred.loop_predictor'

In [ ]:
# Storage estimation: conditional predictor only (no BTB / RAS / indirect).
# Methodology: count all prediction counter/tag arrays for each predictor.
#   BiModeBP  — choice table + takenCounters + notTakenCounters
#   GshareBP  — global PHT + LocalBP counter table
#   TAGE      — base bimodal table (index 0) + all 7 tagged history tables
#   LAPBP     — bimodal base + loop predictor entries + WITHLOOP counter

_SEC_COND = 'system.cpu.branchPred.conditionalBranchPred'
_SEC_BP   = 'system.cpu.branchPred'
_SEC_TAGE = 'system.cpu.branchPred.conditionalBranchPred.tage'
_SEC_LP   = 'system.cpu.branchPred.conditionalBranchPred.loop_predictor'


def estimate_storage_kb(cfg):
    cond_type = _cs(cfg, _SEC_COND, 'type')
    bp_type   = _cs(cfg, _SEC_BP,   'type')

    if cond_type == 'BiModeBP':
        choice = _ci(cfg, _SEC_COND, 'choicePredictorSize') * _ci(cfg, _SEC_COND, 'choiceCtrBits')
        global_ = _ci(cfg, _SEC_COND, 'globalPredictorSize') * _ci(cfg, _SEC_COND, 'globalCtrBits')
        bits   = choice + 2 * global_  # choice + takenCounters + notTakenCounters
        return bits / 8 / 1024

    if cond_type == 'TAGE':
        log_sizes  = _cil(cfg, _SEC_TAGE, 'logTagTableSizes',  default=[])
        tag_widths = _cil(cfg, _SEC_TAGE, 'tagTableTagWidths', default=[])
        ctr   = _ci(cfg, _SEC_TAGE, 'tagTableCounterBits')
        ubits = _ci(cfg, _SEC_TAGE, 'tagTableUBits')
        n    = min(len(log_sizes), len(tag_widths))
        bits = sum((2 ** log_sizes[i]) * (tag_widths[i] + ctr + ubits) for i in range(n))
        return bits / 8 / 1024

    if bp_type == 'GshareBP':
        g = _ci(cfg, _SEC_BP,   'global_predictor_size') * _ci(cfg, _SEC_BP,   'global_counter_bits')
        l = (_ci(cfg, _SEC_COND, 'localPredictorSize') * _ci(cfg, _SEC_COND, 'localCtrBits')
             if cond_type == 'LocalBP' else 0)
        return (g + l) / 8 / 1024

    if cond_type == 'LAPBP':
        bimodal = _ci(cfg, _SEC_COND, 'basePredictorSize') * _ci(cfg, _SEC_COND, 'baseCtrBits')
        n_ent  = 2 ** _ci(cfg, _SEC_LP, 'logSizeLoopPred')
        tag_b  = _ci(cfg, _SEC_LP, 'loopTableTagBits')
        iter_b = _ci(cfg, _SEC_LP, 'loopTableIterBits')
        age_b  = _ci(cfg, _SEC_LP, 'loopTableAgeBits')
        conf_b = _ci(cfg, _SEC_LP, 'loopTableConfidenceBits')
        with_b = _ci(cfg, _SEC_LP, 'withLoopBits')
        # per entry: tag + numIter + currentIter + age + confidence
        loop = n_ent * (tag_b + 2 * iter_b + age_b + conf_b) + with_b
        return (bimodal + loop) / 8 / 1024

    return None

In [ ]:
rows = []
for workload in WORKLOADS:
    for predictor in PREDICTORS:
        run_dir    = RESULTS_DIR / workload / predictor
        stats_path = run_dir / 'stats.txt'
        cfg_path   = run_dir / 'config.ini'

        if not stats_path.exists() or stats_path.stat().st_size == 0:
            print(f'SKIP  {workload}/{predictor} — stats.txt missing or empty')
            continue

        stats = load_stats(stats_path)

        lk = pick_stat(stats, _LOOKUP_KEYS)
        mp = pick_stat(stats, _MISPRED_KEYS)
        mispred_rate = (mp / lk) if (lk and mp and lk > 0) else None

        storage_kb = None
        if cfg_path.exists():
            storage_kb = estimate_storage_kb(load_cfg(cfg_path))

        rows.append({
            'workload':    workload,
            'predictor':   predictor,
            'ipc':         stats.get('system.cpu.ipc'),
            'cpi':         stats.get('system.cpu.cpi'),
            'bp_lookups':  lk,
            'bp_mispreds': mp,
            'mispred_rate':mispred_rate,
            'storage_kb':  storage_kb,
            # Loop predictor stats — non-None only for LAPBP runs
            'lp_used':     stats.get(f'{_LP_PREFIX}.used'),
            'lp_correct':  stats.get(f'{_LP_PREFIX}.correct'),
            'lp_wrong':    stats.get(f'{_LP_PREFIX}.wrong'),
        })

df = pd.DataFrame(rows)
print(f'Loaded {len(df)} runs ({len(WORKLOADS)} workloads x {len(PREDICTORS)} predictors)')
df[['workload', 'predictor', 'ipc', 'mispred_rate', 'storage_kb', 'lp_used', 'lp_correct', 'lp_wrong']]

## Comparison Tables

Storage accounting:  
- **BiModeBP**: choice table + takenCounters + notTakenCounters  
- **GshareBP**: global PHT + LocalBP counter table  
- **TAGE**: base bimodal table + 7 tagged history tables (full storage)  
- **LAP**: bimodal base + loop predictor entries (tag/iter/age/conf) + WITHLOOP counter

In [ ]:
for wl in WORKLOADS:
    sub = df[df['workload'] == wl].set_index('predictor').reindex(PREDICTORS)
    tbl = pd.DataFrame({
        'IPC':           sub['ipc'].map('{:.3f}'.format),
        'CPI':           sub['cpi'].map('{:.3f}'.format),
        'Mispred Rate':  sub['mispred_rate'].map('{:.2%}'.format),
        'Storage (KB)':  sub['storage_kb'].map(lambda x: f'{x:.3f}' if pd.notna(x) else 'N/A'),
    })
    print(f'{'='*54}')
    print(f'  Workload: {wl}')
    print(f'{'='*54}')
    print(tbl.to_string())
    print()

## IPC Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
x       = np.arange(len(WORKLOADS))
n       = len(PREDICTORS)
width   = 0.18
offsets = np.linspace(-(n - 1) / 2, (n - 1) / 2, n) * width

for i, pred in enumerate(PREDICTORS):
    vals = []
    for wl in WORKLOADS:
        row = df[(df['workload'] == wl) & (df['predictor'] == pred)]
        vals.append(float(row['ipc'].values[0]) if len(row) and pd.notna(row['ipc'].values[0]) else np.nan)
    bars = ax.bar(x + offsets[i], vals, width * 0.9, label=pred,
                  color=PRED_COLORS.get(pred, f'C{i}'), zorder=3)
    for bar, val in zip(bars, vals):
        if not np.isnan(val):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.04,
                    f'{val:.2f}', ha='center', va='bottom', fontsize=7.5)

ax.set_xticks(x)
ax.set_xticklabels([wl.upper() for wl in WORKLOADS], fontsize=12)
ax.set_ylabel('IPC (Instructions Per Cycle)', fontsize=10)
ax.set_title('IPC by Branch Predictor and Workload', fontsize=12, fontweight='bold')
ax.legend(fontsize=9, loc='upper left')
ax.set_ylim(0, ax.get_ylim()[1] * 1.16)
ax.yaxis.grid(True, alpha=0.35, zorder=0)
ax.set_axisbelow(True)
fig.tight_layout()
plt.savefig(PROJECT_ROOT / 'results' / 'ipc_comparison.png', dpi=150)
plt.show()

## Misprediction Rate

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
x       = np.arange(len(WORKLOADS))
n       = len(PREDICTORS)
width   = 0.18
offsets = np.linspace(-(n - 1) / 2, (n - 1) / 2, n) * width

for i, pred in enumerate(PREDICTORS):
    vals = []
    for wl in WORKLOADS:
        row = df[(df['workload'] == wl) & (df['predictor'] == pred)]
        mr  = row['mispred_rate'].values[0] if len(row) and pd.notna(row['mispred_rate'].values[0]) else np.nan
        vals.append(float(mr) * 100 if not np.isnan(mr) else np.nan)
    bars = ax.bar(x + offsets[i], vals, width * 0.9, label=pred,
                  color=PRED_COLORS.get(pred, f'C{i}'), zorder=3)
    for bar, val in zip(bars, vals):
        if not np.isnan(val):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.07,
                    f'{val:.1f}%', ha='center', va='bottom', fontsize=7)

ax.set_xticks(x)
ax.set_xticklabels([wl.upper() for wl in WORKLOADS], fontsize=12)
ax.set_ylabel('Misprediction Rate (%)', fontsize=10)
ax.set_title('Misprediction Rate by Branch Predictor and Workload', fontsize=12, fontweight='bold')
ax.legend(fontsize=9, loc='upper right')
ax.set_ylim(0, ax.get_ylim()[1] * 1.18)
ax.yaxis.grid(True, alpha=0.35, zorder=0)
ax.set_axisbelow(True)
fig.tight_layout()
plt.savefig(PROJECT_ROOT / 'results' / 'mispred_comparison.png', dpi=150)
plt.show()

## LABP Loop Predictor Activity — matmul

The loop predictor inside LABP fires only when `loopPredValid` (confidence == max) **and**
`loopUseCounter >= 0`. matmul's triple-nested counted loops are the ideal workload for it.

In [ ]:
lap_rows = df[(df['workload'] == 'matmul') & (df['predictor'] == 'LAP')]

if lap_rows.empty or pd.isna(lap_rows.iloc[0]['lp_used']):
    print('LAP/matmul run not found or loop predictor stats unavailable.')
else:
    row     = lap_rows.iloc[0]
    used    = int(row['lp_used'])
    correct = int(row['lp_correct'])
    wrong   = int(row['lp_wrong'])
    accuracy = correct / used * 100 if used > 0 else 0.0

    fig, ax = plt.subplots(figsize=(6, 4.5))
    labels  = ['Used', 'Correct', 'Wrong']
    values  = [used, correct, wrong]
    colors  = ['#4c72b0', '#55a868', '#c44e52']
    bars    = ax.bar(labels, values, color=colors, width=0.45, zorder=3)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(values) * 0.012,
                f'{val:,}', ha='center', va='bottom', fontsize=11, fontweight='bold')

    ax.set_ylabel('Branch Count', fontsize=10)
    ax.set_title('LABP Loop Predictor Activity — matmul', fontsize=12, fontweight='bold')
    ax.set_xlabel(
        f'Accuracy when used: {accuracy:.1f}%  ({correct:,} correct / {used:,} total)',
        fontsize=10)
    ax.yaxis.grid(True, alpha=0.35, zorder=0)
    ax.set_axisbelow(True)
    ax.set_ylim(0, max(values) * 1.18)
    fig.tight_layout()
    plt.savefig(PROJECT_ROOT / 'results' / 'lap_loop_predictor_matmul.png', dpi=150)
    plt.show()

    print(f'Loop predictor used:    {used:>10,}')
    print(f'Loop predictor correct: {correct:>10,}  ({correct / used * 100:.1f}%)')
    print(f'Loop predictor wrong:   {wrong:>10,}  ({wrong / used * 100:.1f}%)')

    # Also show tinymlp loop stats for comparison
    lap_tiny = df[(df['workload'] == 'tinymlp') & (df['predictor'] == 'LAP')]
    if not lap_tiny.empty and pd.notna(lap_tiny.iloc[0]['lp_used']):
        t = lap_tiny.iloc[0]
        print()
        print(f'tinymlp — lp_used: {int(t["lp_used"]):,}  correct: {int(t["lp_correct"]):,}  wrong: {int(t["lp_wrong"]):,}')

## Storage Efficiency: IPC per KB

Normalises IPC against conditional predictor storage to identify the best accuracy-per-area trade-off.

In [ ]:
df_eff = df.dropna(subset=['storage_kb', 'ipc']).copy()
df_eff['ipc_per_kb'] = df_eff['ipc'] / df_eff['storage_kb']

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, wl in zip(axes, WORKLOADS):
    sub = (df_eff[df_eff['workload'] == wl]
           .set_index('predictor').reindex(PREDICTORS)
           .dropna(subset=['ipc_per_kb']))
    colors = [PRED_COLORS.get(p, 'gray') for p in sub.index]
    bars = ax.bar(list(sub.index), sub['ipc_per_kb'].values, color=colors, zorder=3)
    for bar, val in zip(bars, sub['ipc_per_kb'].values):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + sub['ipc_per_kb'].max() * 0.01,
                f'{val:.2f}', ha='center', va='bottom', fontsize=9)
    ax.set_title(f'IPC / KB of Storage — {wl.upper()}', fontsize=11, fontweight='bold')
    ax.set_ylabel('IPC per KB of cond. predictor storage', fontsize=9)
    ax.yaxis.grid(True, alpha=0.35, zorder=0)
    ax.set_axisbelow(True)
    ax.set_xticklabels(list(sub.index), rotation=20, ha='right')
    ax.set_ylim(0, sub['ipc_per_kb'].max() * 1.18)

fig.suptitle('Storage Efficiency: IPC per KB of Conditional Predictor Storage',
             fontsize=12, fontweight='bold')
fig.tight_layout()
plt.savefig(PROJECT_ROOT / 'results' / 'storage_efficiency.png', dpi=150)
plt.show()

print('\nIPC / KB table:')
pivot = (df_eff
         .pivot(index='predictor', columns='workload', values='ipc_per_kb')
         .reindex(PREDICTORS))
pivot.columns.name = None
print(pivot.to_string(float_format='{:.3f}'.format))

## Export

In [ ]:
out = RESULTS_DIR / 'all_predictor_comparison.csv'
df.to_csv(out, index=False)
print('Saved:', out)
print()
print(df[['workload', 'predictor', 'ipc', 'cpi', 'mispred_rate', 'storage_kb']].to_string(index=False))